# Adjusted Cosine Similarity

**Medium** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `Recommender Systems`

Adjusted cosine similarity is used in **item-based collaborative filtering** to
measure the similarity between two items. Unlike regular cosine similarity, it
accounts for differences in user rating scales by subtracting each user's mean
rating before computing the similarity.

A user who rates everything 4–5 and a user who rates 1–2 should contribute
equally, which raw cosine similarity fails to capture.

Given a ratings matrix (users × items, `0` = unrated), compute the adjusted
cosine similarity between two specified items. **Only consider users who have
rated both items.**

$$\text{sim}(i, j) = \frac{\sum_{u \in U_{ij}} (r_{ui} - \bar{r}_u)(r_{uj} - \bar{r}_u)}
{\sqrt{\sum_{u \in U_{ij}} (r_{ui} - \bar{r}_u)^2} \cdot \sqrt{\sum_{u \in U_{ij}} (r_{uj} - \bar{r}_u)^2}}$$

where $U_{ij}$ is the set of users who rated **both** items, and $\bar{r}_u$ is
the mean of user `u`'s non-zero ratings.

---

**Example 1:**

```
Input:  ratings = [[5, 3], [4, 0], [0, 1], [0, 4]], item_i = 0, item_j = 1
Output: -1.0
Only user 0 rated both items. User 0 mean = (5+3)/2 = 4.
Centered: (5-4, 3-4) = (1, -1). Similarity = (1)(-1) / ((1)(1)) = -1.0
```

**Example 2:**

```
Input:  ratings = [[5, 1], [4, 2], [3, 3]], item_i = 0, item_j = 1
Output: -1.0
All users rated both items. User means: 3, 3, 3.
Centered item 0: (2, 1, 0). Centered item 1: (-2, -1, 0).
Perfect negative correlation.
```

---

**Hint 1:** first compute each user's mean rating (only over non-zero entries).
Then for each user who rated both items, compute the centered ratings and
accumulate the numerator and denominator terms.

**Hint 2:** the denominator has two parts under separate square roots multiplied
together. Track the sum of squared centered ratings for each item separately,
then take the product of their square roots.

**Requirements:**

- compute per-user mean ratings using only non-zero (rated) entries
- only include users who have rated **both** items in the similarity calculation
- center each rating by subtracting the user mean before computing similarity
- return `0.0` if the denominator is zero **or** no users rated both items

**Constraints:**

- `ratings_matrix` is a list of lists (users × items), `0` means unrated
- `item_i` and `item_j` are valid column indices
- return a float between `-1.0` and `1.0`
- time limit 300 ms

### Three places this goes wrong

This is the Medium one, and it earns it. The formula is not the hard part — the
bookkeeping around it is. Three traps, in the order you will hit them:

**1. The user means.** `R.mean(axis=1)` is wrong, and it will not crash, it will
just quietly give you bad numbers. `0` means *unrated*, not *rated zero*, so
averaging it in drags every mean down. You need the mean over non-zero entries
only. Think about how to count and sum only the entries where a mask is true —
and what to do about a user who rated nothing at all, so you do not divide by
zero building the means.

**2. Which users count.** Only users who rated **both** items. Build that mask
explicitly. If nobody qualifies, the answer is `0.0`.

**3. Which mean, over which users.** Read the formula carefully — this is the
subtle one. $\bar{r}_u$ is the mean of user `u`'s ratings across the **whole
row**, every item they rated. But the three sums only run over $U_{ij}$, the
co-rating users. So the mean is computed on one set of entries and applied to a
different, smaller set. Do not accidentally take the mean of just the two
columns.

Then the guard: if the denominator is `0` (a user whose two centered ratings are
both zero contributes nothing), return `0.0`.

### Write it twice

Write the plain Python loop version first — it maps line by line onto the formula
and you can debug it by printing. *Then* vectorize it with NumPy.

The test cell checks the two against each other on 64 random item pairs, which is
worth far more than the handful of hand-picked cases: if you got a mask subtly
wrong in one version, the two will disagree somewhere in there.

### And a third, if you want to see the point of it

`similarity_matrix(ratings)` — every item against every other item, which is what
a real item-based recommender computes once and then just looks up. The result is
symmetric, so you only need to compute half of it.

In [3]:
import numpy as np


class Solution:
    def normalize(self,x):
        nc = 0
        s = 0.0
        for i in x:
            if i != 0 :
                nc +=1
                s += i
        return s/nc
    def dot_product(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")
        return float(np.dot(x, y))

    def norm(self,a):
        return np.sqrt(np.dot(a, a))

    def cosine_similarity(self, a, b) -> float:
        a_norm = self.norm(a)
        b_norm = self.norm(b)
        if a_norm == 0 or b_norm == 0:
            return 0.0
        return self.dot_product(a,b)/(a_norm*b_norm)

    def adjusted_cosine_similarity(self, ratings_matrix, item_i, item_j):
        i_normalized = []
        j_normalized = []

        for user in ratings_matrix:
            user_mean = self.normalize(user)
            rating_i = user[item_i]
            rating_j = user[item_j]
            if rating_i != 0 and rating_j != 0:
                i_normalized.append(rating_i - user_mean)
                j_normalized.append(rating_j - user_mean)
        return self.cosine_similarity(i_normalized, j_normalized)


    # write this one FIRST - it reads like the formula and is easy to debug
    def adjusted_cosine_loops(self, ratings_matrix, item_i, item_j) -> float:
        i_normalized = []
        j_normalized = []

        for user in ratings_matrix:
            user_mean = self.normalize(user)
            rating_i = user[item_i]
            rating_j = user[item_j]
            if rating_i != 0 and rating_j != 0:
                i_normalized.append(rating_i - user_mean)
                j_normalized.append(rating_j - user_mean)
        return self.cosine_similarity(i_normalized, j_normalized)

    # optional: the full item x item matrix
    def similarity_matrix(self, ratings_matrix):
        n_items = len(ratings_matrix[0])
        matrix = np.zeros((n_items, n_items))
        skiped=1
        for i , j in enumerate(ratings_matrix):
            for k , l in enumerate(j):
                if k+skiped >= len(j): break
                elif i == k : matrix[i][i] = 1
                else :
                    cos = self.adjusted_cosine_similarity(ratings_matrix,i,k+skiped)
                    matrix[i][k] = cos
                    matrix[k][i] = cos
            skiped += 1
        return matrix






In [4]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


sol = Solution()

cases = [
    # (ratings, i, j, expected)
    ([[5, 3], [4, 0], [0, 1], [0, 4]], 0, 1, -1.0),   # example 1: one co-rating user
    ([[5, 1], [4, 2], [3, 3]],         0, 1, -1.0),   # example 2: perfect negative
    ([[5, 3, 0], [4, 0, 2], [0, 1, 4]], 0, 1, -1.0),  # from the problem screenshot
    ([[5, 5, 1], [1, 1, 5]],           0, 1,  1.0),   # both items move with the baseline
    ([[1, 2], [2, 4], [3, 6]],         0, 1, -1.0),   # see the note below - NOT +1.0
    ([[5, 0], [0, 3]],                 0, 1,  0.0),   # nobody rated both -> 0.0
    ([[3, 3], [4, 4]],                 0, 1,  0.0),   # flat raters: denom 0 -> 0.0
]

print("adjusted_cosine_loops")
for R, i, j, want in cases:
    print(f"  {str(R):<36} -> {check(sol.adjusted_cosine_loops(R, i, j), want)}")

print("\nadjusted_cosine_similarity")
for R, i, j, want in cases:
    print(f"  {str(R):<36} -> {check(sol.adjusted_cosine_similarity(R, i, j), want)}")

# the real test: do the two versions agree everywhere, not just on these 7?
rng = np.random.default_rng(7)
R = rng.integers(0, 6, size=(40, 8)).astype(float)     # 0 = unrated, ~17% of cells
try:
    diffs = [abs(sol.adjusted_cosine_similarity(R, i, j) - sol.adjusted_cosine_loops(R.tolist(), i, j))
             for i in range(8) for j in range(8)]
    print(f"\nboth versions agree on 64 random pairs: {max(diffs) < 1e-9}")
    S = sol.similarity_matrix(R)
    print("similarity_matrix stays within [-1, 1]:", bool(np.all(np.abs(S) <= 1 + 1e-9)))
    print("similarity_matrix is symmetric        :", bool(np.allclose(S, S.T)))
except TypeError:
    print("\n(finish both versions to run the cross-check)")

adjusted_cosine_loops
  [[5, 3], [4, 0], [0, 1], [0, 4]]     -> OK
  [[5, 1], [4, 2], [3, 3]]             -> OK
  [[5, 3, 0], [4, 0, 2], [0, 1, 4]]    -> OK
  [[5, 5, 1], [1, 1, 5]]               -> OK
  [[1, 2], [2, 4], [3, 6]]             -> OK
  [[5, 0], [0, 3]]                     -> OK
  [[3, 3], [4, 4]]                     -> OK

adjusted_cosine_similarity
  [[5, 3], [4, 0], [0, 1], [0, 4]]     -> OK
  [[5, 1], [4, 2], [3, 3]]             -> OK
  [[5, 3, 0], [4, 0, 2], [0, 1, 4]]    -> OK
  [[5, 5, 1], [1, 1, 5]]               -> OK
  [[1, 2], [2, 4], [3, 6]]             -> OK
  [[5, 0], [0, 3]]                     -> OK
  [[3, 3], [4, 4]]                     -> OK

both versions agree on 64 random pairs: True
similarity_matrix stays within [-1, 1]: True
similarity_matrix is symmetric        : True


### The case that looks like `+1.0` and is not

`[[1, 2], [2, 4], [3, 6]]` — item `j` is exactly twice item `i` for every single
user. Raw cosine similarity calls that a perfect `+1.0`. The expected answer in
the test is **`-1.0`**.

Do not "fix" your code to make that come out positive. Work out why `-1.0` is
correct: centering happens per **user**, and each user's mean lands *between*
their two ratings. So whichever item is the lower one sits below the baseline
while the other sits above it — for every user, without exception.

Write the three centered pairs out by hand and you will see it immediately. Then
notice the general fact hiding there: with only two items in the matrix and no
ties, the answer is *always* `-1.0`.

That is not a bug in the metric. It is measuring "do users who like item `i` more
than their own average also like item `j` more than their own average" — which is
a different question from "do these two items get similar scores".

### Then: why bother centering at all

The last cell has three users and two items, one of whom rates everything
generously. Compare raw cosine against your adjusted version and see which one
notices that the users actually disagree.

In [ ]:
# user 0 is a generous rater (5,4), user 1 is harsh (2,1), user 2 disagrees with both
ratings = [
    [5, 4],
    [2, 1],
    [1, 5],
]

def raw_cosine(R, i, j):
    R = np.asarray(R, dtype=float)
    a, b = R[:, i], R[:, j]
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"raw cosine       = {raw_cosine(ratings, 0, 1): .4f}")
print(f"adjusted cosine  = {sol.adjusted_cosine_similarity(ratings, 0, 1)}")
print()

R = np.asarray(ratings, float)
means = R.sum(1) / (R != 0).sum(1)
print("user means (non-zero only):", means.round(3))
print("centered item 0:", (R[:, 0] - means).round(3))
print("centered item 1:", (R[:, 1] - means).round(3))